<a href="https://colab.research.google.com/github/saiteja-1919/RAG-VS-NON-RAG-MODEL/blob/main/NLP_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pdfplumber sentence-transformers faiss-cpu transformers torch torchvision torchaudio accelerate reportlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 85.3 MB/s eta 0:00:00


In [9]:
pip install pypdf sentence-transformers faiss-cpu numpy transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.2/331.2 kB 7.5 MB/s eta 0:00:00


In [ ]:
# ================================================================
#   RAG vs Non-RAG Comparison
#   PDF-based Question Answering using FAISS + Flan-T5
# ================================================================

# ---------- Imports ----------
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ================================================================
#  SECTION 1 — PDF INGESTION
# ================================================================

PDF_PATH = "/content/MACHINE LEARNING(R17A0534) (1).pdf"

def read_pdf(filepath):
    """Extract all text from a PDF file."""
    reader = PdfReader(filepath)
    full_text = ""
    for pg in reader.pages:
        extracted = pg.extract_text()
        if extracted:
            full_text += extracted + "\n"
    return full_text

print(">>> Reading PDF...")
raw_text = read_pdf(PDF_PATH)
print(f">>> Done! Total characters extracted: {len(raw_text)}\n")


# ================================================================
#  SECTION 2 — TEXT CHUNKING
# ================================================================

def split_into_chunks(text, chunk_size=220, overlap=40):
    """
    Split text into overlapping word-based chunks.
    Overlap helps preserve context at chunk boundaries.
    """
    words = text.split()
    step = chunk_size - overlap
    segments = []
    for start in range(0, len(words), step):
        segment = " ".join(words[start : start + chunk_size])
        segments.append(segment)
    return segments

text_chunks = split_into_chunks(raw_text)
print(f">>> Total chunks created: {len(text_chunks)}\n")


# ================================================================
#  SECTION 3 — EMBEDDING GENERATION
# ================================================================

print(">>> Generating embeddings...")
encoder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = encoder.encode(text_chunks, show_progress_bar=True)
print(">>> Embeddings ready!\n")


# ================================================================
#  SECTION 4 — VECTOR INDEX (FAISS)
# ================================================================

vec_dim = chunk_vectors.shape[1]
vector_store = faiss.IndexFlatL2(vec_dim)
vector_store.add(np.array(chunk_vectors))
print(f">>> FAISS index built with {vector_store.ntotal} vectors\n")


# ================================================================
#  SECTION 5 — LANGUAGE MODEL SETUP
# ================================================================

MODEL_NAME = "google/flan-t5-base"

print(f">>> Loading language model: {MODEL_NAME}...")
lm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
lm_model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(">>> Language model loaded!\n")

def run_model(prompt_text, max_tokens=200):
    """Tokenize the prompt and return generated answer."""
    tokens = lm_tokenizer(prompt_text, return_tensors="pt", truncation=True)
    output = lm_model.generate(**tokens, max_new_tokens=max_tokens)
    return lm_tokenizer.decode(output[0], skip_special_tokens=True)


# ================================================================
#  SECTION 6 — RAG PIPELINE
# ================================================================

def rag_answer(query, top_k=3, context_limit=1200):
    """
    Retrieval-Augmented Generation:
    1. Embed the query
    2. Retrieve top-k relevant chunks from FAISS
    3. Build a grounded prompt and generate an answer
    """
    query_vec = encoder.encode([query])
    _, matched_ids = vector_store.search(np.array(query_vec), k=top_k)

    retrieved_context = " ".join(
        [text_chunks[idx] for idx in matched_ids[0]]
    )[:context_limit]

    prompt = (
        "You are a helpful AI tutor explaining concepts from a machine learning textbook.\n"
        "Use the provided context to give a clear, complete answer in 3 to 5 sentences.\n"
        "Use simple language.\n\n"
        f"Context:\n{retrieved_context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return run_model(prompt).strip()


# ================================================================
#  SECTION 7 — NON-RAG PIPELINE
# ================================================================

def plain_answer(query):
    """
    Non-RAG baseline:
    Directly feeds the question to the model with no retrieved context.
    """
    prompt = (
        "You are a helpful AI tutor with knowledge of machine learning.\n"
        "Answer the following question in 3 to 5 sentences using simple language.\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return run_model(prompt).strip()


# ================================================================
#  SECTION 8 — INTERACTIVE COMPARISON LOOP
# ================================================================

print("=" * 60)
print("   RAG vs Non-RAG — Interactive Q&A")
print("   Type 'exit' to quit")
print("=" * 60 + "\n")

while True:
    user_query = input("Your Question: ").strip()
    if user_query.lower() == "exit":
        print("Goodbye!")
        break
    if not user_query:
        continue

    print("\n[WITHOUT RAG]")
    print(plain_answer(user_query))

    print("\n[WITH RAG]")
    print(rag_answer(user_query))

    print("\n" + "-" * 60 + "\n")

>>> Reading PDF...
>>> Done! Total characters extracted: 243843

>>> Total chunks created: 221

>>> Generating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

>>> Embeddings ready!

>>> FAISS index built with 221 vectors

>>> Loading language model: google/flan-t5-base...


...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

>>> Language model loaded!

   RAG vs Non-RAG — Interactive Q&A
   Type 'exit' to quit

Your Question: what is supervised learning

[WITHOUT RAG]
supervised learning is a type of learning that is done by a trained human.

[WITH RAG]
a function that maps an input to an output based on example input-output pairs

------------------------------------------------------------

